In [ ]:
# notebook to look at examples of emvars in tf seqlets within the same enhancer

In [1]:
# import packages
import pandas as pd
import os
import pybedtools
from tqdm import tqdm
from collections import Counter

In [2]:
def multiTF_analysis(path2distalCREpreds, 
                     path2distalCREseqlets_k562,
                     path2distalCREseqlets_hepg2,
                     path2distalCREseqlets_sknsh,
                     vierstra_dict,
                     read_all_chunks=False,
                     n_test_chunks=2,  # Reduced for faster testing
                     max_overlap_bp=2):
    """
    Analyze multi-TF enhancers across cell types and generate variant-level dataframes.
    
    Args:
        n_test_chunks: Number of chunks to read when read_all_chunks=False (default 2 for faster testing)
        max_overlap_bp: Maximum allowed overlap between seqlets from same vierstra cluster (default 2bp)
    
    Returns:
        tuple: (raw_data, multiTF_summary)
    """
    # Helper: read MPAC predictions in chunks
    def read_in_sat_mut(path2satmut, chunksize=1000000, read_all=False, n_chunks=2):
        chunks2cat = []
        count = 0
        for chunk in tqdm(pd.read_csv(path2satmut, sep='\t', chunksize=chunksize)):
            chunks2cat.append(chunk)
            count += 1
            if not read_all and count >= n_chunks:
                break
        return pd.concat(chunks2cat)
    
    # Helper: convert emVar DF to BED (vectorized)
    def df2bed(emvar_df):
        df = emvar_df.copy()
        df['bed_id'] = df['chrom'] + ':' + df['pos'].astype(str) + ':' + df['ref'] + ':' + df['alt'] + '_' + df['id']
        
        df['bed_start'] = df['pos'] - 1
        df['bed_end'] = df['pos']
        
        ins_mask = df['ref'].str.len() < df['alt'].str.len()
        df.loc[ins_mask, 'bed_start'] = df.loc[ins_mask, 'pos']
        df.loc[ins_mask, 'bed_end'] = df.loc[ins_mask, 'pos']
        
        del_mask = df['ref'].str.len() > df['alt'].str.len()
        df.loc[del_mask, 'bed_start'] = df.loc[del_mask, 'pos']
        df.loc[del_mask, 'bed_end'] = df.loc[del_mask, 'pos'] + df.loc[del_mask, 'ref'].str.len() - 1
        
        bed_df = df[['chrom', 'bed_start', 'bed_end', 'bed_id']].drop_duplicates()
        bed_df.columns = [0, 1, 2, 3]
        return pybedtools.BedTool.from_dataframe(bed_df).sort()
    
    # Helper: create long-form TF dataframe with collapse to representative TF per interval
    def create_long_form_tf_df(raw_bed_df, vierstra_dict):
        raw_bed_df = raw_bed_df.copy()
        raw_bed_df[3] = raw_bed_df['name'].astype(str)
        df_split = raw_bed_df.assign(tf_hits_list=raw_bed_df[3].str.split(';'))
        long_df = df_split.explode('tf_hits_list').reset_index(drop=True)
        
        long_df['full_hit_string'] = long_df['tf_hits_list']
        long_df['hocomoco_tf'] = long_df['full_hit_string'].str.split('_EH').str[0]
        long_df['representative_tf'] = long_df['hocomoco_tf'].str.split('_').str[0]
        long_df['enhancer_id'] = long_df['full_hit_string'].str.split('_').str[2]
        
        long_df['rep_tf_contrib'] = pd.to_numeric(
            long_df['full_hit_string'].str.split('_').str[-1], errors='coerce'
        ).fillna(0.0)
        
        long_df['vierstra_cluster'] = long_df['hocomoco_tf'].map(vierstra_dict)
        long_df['activity_class'] = pd.cut(
            long_df['rep_tf_contrib'], 
            bins=[-float('inf'), 0, float('inf')], 
            labels=['Repressor', 'Activator']
        ).astype(str)
        long_df.loc[long_df['rep_tf_contrib'] == 0, 'activity_class'] = 'Neutral'
        
        long_df = long_df.drop(columns=['tf_hits_list', 3])
        
        # Collapse to representative TF per interval (highest |contribution|)
        long_df['abs_contrib'] = long_df['rep_tf_contrib'].abs()
        idx_max = long_df.groupby(['chrom', 'start', 'end'])['abs_contrib'].idxmax()
        collapsed_df = long_df.loc[idx_max].drop(columns=['abs_contrib']).reset_index(drop=True)
        
        return collapsed_df
    
    # Helper: get multi-TF info per enhancer (OPTIMIZED - uses groupby iteration)
    def get_multiTF_info(collapsed_seqlets, max_overlap_bp):
        """Returns dict mapping enhancer_id -> list of vierstra clusters with >1 non-overlapping instance."""
        valid_clusters = {}
        
        # Group once by enhancer_id and vierstra_cluster, then iterate over groups
        for (enh_id, cluster), group in collapsed_seqlets.groupby(['enhancer_id', 'vierstra_cluster']):
            if len(group) < 2 or pd.isna(cluster):
                continue
            
            # Check if seqlets are non-overlapping (sorted check)
            sorted_group = group.sort_values('start')
            is_valid = True
            prev_end = None
            for _, row in sorted_group.iterrows():
                if prev_end is not None:
                    overlap = prev_end - row['start']
                    if overlap > max_overlap_bp:
                        is_valid = False
                        break
                prev_end = row['end']
            
            if is_valid:
                if enh_id not in valid_clusters:
                    valid_clusters[enh_id] = []
                valid_clusters[enh_id].append(cluster)
        
        return valid_clusters
    
    # Helper: process emvar_seqlets to add variant_id and merge predictions
    def process_emvar_seqlets(emvar_seqlets_df, mpac_df, cell_type):
        if len(emvar_seqlets_df) == 0:
            return emvar_seqlets_df
        
        df = emvar_seqlets_df.copy()
        df['variant_id'] = df['blockStarts'].str.split('_').str[0]
        
        variant_parts = df['variant_id'].str.split(':', expand=True)
        df['var_chrom'] = variant_parts[0]
        df['var_pos'] = variant_parts[1].astype(int)
        df['var_ref'] = variant_parts[2]
        df['var_alt'] = variant_parts[3]
        
        skew_col = f'{cell_type}_skew_pred'
        mpac_subset = mpac_df[['chrom', 'pos', 'ref', 'alt', skew_col]].copy()
        mpac_subset = mpac_subset.rename(columns={skew_col: 'skew_pred'})
        
        df = df.merge(
            mpac_subset,
            left_on=['var_chrom', 'var_pos', 'var_ref', 'var_alt'],
            right_on=['chrom', 'pos', 'ref', 'alt'],
            how='left',
            suffixes=('', '_mpac')
        )
        df = df.drop(columns=['chrom_mpac', 'pos_mpac', 'ref_mpac', 'alt_mpac'], errors='ignore')
        
        return df
    
    # Helper: create variant dataframe
    def create_variant_dataframe(emvar_seqlets_df, multiTF_enhancer_ids, cell_type):
        if len(emvar_seqlets_df) == 0:
            return pd.DataFrame()
        
        df = emvar_seqlets_df.copy()
        
        variant_agg = df.groupby('variant_id').agg({
            'thickEnd': lambda x: ','.join(sorted(set(x))),
            'var_chrom': 'first',
            'var_pos': 'first',
            'var_ref': 'first',
            'var_alt': 'first',
            'skew_pred': 'first'
        }).reset_index()
        variant_agg.columns = ['variant_id', 'enhancer_ids', 'chrom', 'pos', 'ref', 'alt', 'skew_pred']
        
        def check_multiTF(enh_str):
            return any(e in multiTF_enhancer_ids for e in enh_str.split(','))
        variant_agg['is_multiTF'] = variant_agg['enhancer_ids'].apply(check_multiTF)
        variant_agg['cell_type'] = cell_type
        
        return variant_agg[['variant_id', 'chrom', 'pos', 'ref', 'alt', 'cell_type', 
                          'skew_pred', 'enhancer_ids', 'is_multiTF']]
    
    # === STEP 1: Read MPAC predictions ===
    print(f"Reading MPAC predictions ({'all' if read_all_chunks else f'{n_test_chunks} chunks'})...")
    allPreds = read_in_sat_mut(path2distalCREpreds, read_all=read_all_chunks, n_chunks=n_test_chunks)
    print(f"  Loaded {len(allPreds):,} variants")
    
    # === STEP 2: Filter for emVars in each cell type ===
    print("Filtering for emVars...")
    emvars = {
        'k562': allPreds[allPreds['k562_skew_pred'].abs() > 0.5].copy(),
        'hepg2': allPreds[allPreds['hepg2_skew_pred'].abs() > 0.5].copy(),
        'sknsh': allPreds[allPreds['sknsh_skew_pred'].abs() > 0.5].copy()
    }
    
    # === STEP 3: Convert to BED ===
    print("Converting emVars to BED...")
    emvar_beds = {ct: df2bed(df) for ct, df in emvars.items()}
    
    # === STEP 4: Load seqlet BED files ===
    print("Loading seqlet BED files...")
    seqlet_beds = {
        'k562': pybedtools.BedTool(path2distalCREseqlets_k562),
        'hepg2': pybedtools.BedTool(path2distalCREseqlets_hepg2),
        'sknsh': pybedtools.BedTool(path2distalCREseqlets_sknsh)
    }
    
    # === STEP 5: Process each cell type ===
    raw_data = {}
    multiTF_summary = {}
    
    for cell_type in ['k562', 'hepg2', 'sknsh']:
        print(f"\nProcessing {cell_type}...")
        
        print(f"  Intersecting emVars with seqlets...")
        emvar_seqlets_raw = seqlet_beds[cell_type].intersect(
            emvar_beds[cell_type], wa=True, wb=True
        ).to_dataframe()
        
        print(f"  Processing emvar_seqlets...")
        emvar_seqlets = process_emvar_seqlets(emvar_seqlets_raw, emvars[cell_type], cell_type)
        
        print(f"  Creating collapsed seqlets...")
        collapsed_seqlets = create_long_form_tf_df(seqlet_beds[cell_type].to_dataframe(), vierstra_dict)
        
        print(f"  Finding multi-TF enhancers (max {max_overlap_bp}bp overlap)...")
        multiTF_info = get_multiTF_info(collapsed_seqlets, max_overlap_bp)
        multiTF_enhancer_ids = set(multiTF_info.keys())
        
        print(f"  Creating variant dataframe...")
        variant_df = create_variant_dataframe(emvar_seqlets, multiTF_enhancer_ids, cell_type)
        
        raw_data[cell_type] = {
            'collapsed_seqlets': collapsed_seqlets,
            'emvar_seqlets': emvar_seqlets,
            'emvars': emvars[cell_type],
            'variants': variant_df,
            'multiTF_info': multiTF_info
        }
        
        print(f"  Building summary...")
        emvar_enhancer_ids = set(emvar_seqlets['thickEnd'].unique()) if len(emvar_seqlets) > 0 else set()
        multiTF_with_emvars = multiTF_enhancer_ids & emvar_enhancer_ids
        
        emvar_counts = emvar_seqlets.groupby('thickEnd').size().to_dict()
        
        multiTF_seqlets = collapsed_seqlets[collapsed_seqlets['enhancer_id'].isin(multiTF_with_emvars)]
        seqlet_stats = multiTF_seqlets.groupby('enhancer_id').agg({
            'chrom': 'first',
            'start': 'min', 
            'end': 'max',
            'enhancer_id': 'size'
        }).rename(columns={'enhancer_id': 'n_seqlets'})
        
        summary_rows = []
        for enhancer_id in multiTF_with_emvars:
            stats = seqlet_stats.loc[enhancer_id] if enhancer_id in seqlet_stats.index else {}
            summary_rows.append({
                'enhancer_id': enhancer_id,
                'multiTF_clusters': ','.join(sorted(multiTF_info.get(enhancer_id, []))),
                'n_multiTF_clusters': len(multiTF_info.get(enhancer_id, [])),
                'n_seqlets': stats.get('n_seqlets', 0),
                'n_emvars': emvar_counts.get(enhancer_id, 0),
                'chrom': stats.get('chrom'),
                'start': stats.get('start'),
                'end': stats.get('end')
            })
        
        multiTF_summary[cell_type] = pd.DataFrame(summary_rows)
        if len(multiTF_summary[cell_type]) > 0:
            multiTF_summary[cell_type] = multiTF_summary[cell_type].sort_values('n_emvars', ascending=False).reset_index(drop=True)
        
        print(f"  Total intervals (after collapse): {len(collapsed_seqlets):,}")
        print(f"  Multi-TF enhancers (non-overlapping): {len(multiTF_enhancer_ids):,}")
        print(f"  Multi-TF enhancers with emVars: {len(multiTF_with_emvars):,}")
        print(f"  Variants in output: {len(variant_df):,}")
    
    return raw_data, multiTF_summary


def get_enhancer_data(raw_data, cell_type, enhancer_id):
    """Helper to get detailed data for a specific enhancer on-demand."""
    collapsed = raw_data[cell_type]['collapsed_seqlets']
    emvars = raw_data[cell_type]['emvar_seqlets']
    multiTF_info = raw_data[cell_type].get('multiTF_info', {})
    
    return {
        'collapsed_bed': collapsed[collapsed['enhancer_id'] == enhancer_id],
        'enhancer_emvars': emvars[emvars['thickEnd'] == enhancer_id],
        'multiTF_clusters': multiTF_info.get(enhancer_id, [])
    }

In [3]:
# open vierstra clusters for collapsing on families instead of tfs
vierstra_motifs = pd.read_excel('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/gnomad_buffering_analysis/motif_annotations.xlsx', sheet_name=[0,1])
# make a dictionary out of the clusterIDs and Names - this will be used to generate the final dictionary with the motif names
idName_dict = dict(zip(vierstra_motifs[0]['Cluster_ID'], vierstra_motifs[0]['Name']))
# open the second page and assign the Names to the individual motifs
vierstra_motifs[1].loc[:,'cluster_name'] = [idName_dict.get(i) for i in vierstra_motifs[1]['Cluster_ID']]
# make a dictionary that pulls the motif name as a key and returns the vierstra family as a value
vierstra_motif_dict = dict(zip(vierstra_motifs[1]['Motif'], vierstra_motifs[1]['cluster_name']))

In [4]:
# Run the analysis (read_all_chunks=False for testing, True for full analysis)
raw_data, multiTF_summary = multiTF_analysis(
    '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/mpac_preds/GRCh38-dELS-chr1-ALL-mpac-017.tsv.gz', 
    '../../../processed_data/bed_files/repTF_k562_dELS_seqlets_01.bed', 
    '../../../processed_data/bed_files/repTF_hepg2_dELS_seqlets_01.bed', 
    '../../../processed_data/bed_files/repTF_sknsh_dELS_seqlets_01.bed',
    vierstra_motif_dict,
    read_all_chunks=False  # Set to True for full analysis
)

Reading MPAC predictions (2 chunks)...


0it [00:00, ?it/s]

1it [00:06,  6.95s/it]


  Loaded 2,000,000 variants
Filtering for emVars...
Converting emVars to BED...
Loading seqlet BED files...

Processing k562...
  Intersecting emVars with seqlets...
  Processing emvar_seqlets...
  Creating collapsed seqlets...
  Finding multi-TF enhancers (max 2bp overlap)...
  Creating variant dataframe...
  Building summary...
  Total intervals (after collapse): 1,373,171
  Multi-TF enhancers (non-overlapping): 61,984
  Multi-TF enhancers with emVars: 119
  Variants in output: 17,850

Processing hepg2...
  Intersecting emVars with seqlets...
  Processing emvar_seqlets...
  Creating collapsed seqlets...
  Finding multi-TF enhancers (max 2bp overlap)...
  Creating variant dataframe...
  Building summary...
  Total intervals (after collapse): 1,523,603
  Multi-TF enhancers (non-overlapping): 62,510
  Multi-TF enhancers with emVars: 109
  Variants in output: 14,464

Processing sknsh...
  Intersecting emVars with seqlets...
  Processing emvar_seqlets...
  Creating collapsed seqlets...
  

In [5]:
# Access raw data - variant-level dataframe for K562
k562_variants = raw_data['k562']['variants']
print(f"K562 variants in TF seqlets: {len(k562_variants)}")
print(f"Variants in multi-TF enhancers: {k562_variants['is_multiTF'].sum()}")
k562_variants.head(10)

K562 variants in TF seqlets: 17850
Variants in multi-TF enhancers: 3589


,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF
0,chr1:235709267:G:C,chr1,235709267,G,C,k562,1.285931,EH38E2875608,False
1,chr1:235715990:T:G,chr1,235715990,T,G,k562,-0.871161,EH38E3996731,False
2,chr1:235715991:G:A,chr1,235715991,G,A,k562,-1.030704,EH38E3996731,False
3,chr1:235715991:G:C,chr1,235715991,G,C,k562,-1.026897,EH38E3996731,False
4,chr1:235715991:G:T,chr1,235715991,G,T,k562,-1.039617,EH38E3996731,False
5,chr1:235715992:A:C,chr1,235715992,A,C,k562,-0.943775,EH38E3996731,False
6,chr1:235715992:A:G,chr1,235715992,A,G,k562,-0.963697,EH38E3996731,False
7,chr1:235715992:A:T,chr1,235715992,A,T,k562,-0.926797,EH38E3996731,False
8,chr1:235715993:T:A,chr1,235715993,T,A,k562,-0.698904,EH38E3996731,False
9,chr1:235715993:T:C,chr1,235715993,T,C,k562,-0.916259,EH38E3996731,False


In [6]:
# Summary dataframe - quickly find enhancers by multiTF cluster
# Sorted by n_emvars (most emVars first)
print("K562 Multi-TF Enhancer Summary (sorted by n_emvars):")
multiTF_summary['k562'].head(20)

K562 Multi-TF Enhancer Summary (sorted by n_emvars):


,enhancer_id,multiTF_clusters,n_multiTF_clusters,n_seqlets,n_emvars,chrom,start,end
0,EH38E2875836,"AP1/1,Ebox/CACCTG",2,13,248,chr1,236097147,236097402
1,EH38E1335673,CREB/ATF/3,1,7,118,chr1,33120591,33120755
2,EH38E1335881,GFI,1,7,92,chr1,33341303,33341527
3,EH38E2879773,ETS/2,1,6,85,chr1,247407252,247407425
4,EH38E2810496,AP1/1,1,5,82,chr1,49351905,49352143
5,EH38E1336197,"Ebox/CACCTG,GATA",2,6,81,chr1,33902681,33902785
6,EH38E3960771,"CREB/ATF/3,ETS/2",2,7,79,chr1,33190804,33190926
7,EH38E2875903,GFI,1,6,74,chr1,236163498,236163626
8,EH38E2802090,"AP1/1,HD/15",2,7,74,chr1,34268356,34268489
9,EH38E2801582,HIC/2,1,7,72,chr1,33119894,33120133


In [7]:
# Filter summary to find enhancers with a specific TF family
# Example: find all enhancers with multiple GATA instances
tf_family_of_interest = 'GATA'
k562_filtered = multiTF_summary['k562'][
    multiTF_summary['k562']['multiTF_clusters'].str.contains(tf_family_of_interest, na=False)
]
print(f"Enhancers with multiple {tf_family_of_interest} instances: {len(k562_filtered)}")
k562_filtered.head(10)

Enhancers with multiple GATA instances: 8


,enhancer_id,multiTF_clusters,n_multiTF_clusters,n_seqlets,n_emvars,chrom,start,end
5,EH38E1336197,"Ebox/CACCTG,GATA",2,6,81,chr1,33902681,33902785
15,EH38E2810601,"Ebox/CACCTG,GATA",2,6,64,chr1,50265469,50265717
19,EH38E1435041,"GATA,YY1",2,6,60,chr1,236510090,236510282
27,EH38E2810579,GATA,1,7,52,chr1,50139492,50139655
30,EH38E1434656,GATA,1,5,47,chr1,235937276,235937455
31,EH38E2879564,GATA,1,3,47,chr1,246857123,246857215
34,EH38E2810584,GATA,1,2,43,chr1,50158254,50158280
47,EH38E1335816,GATA,1,3,33,chr1,33275245,33275502


In [18]:
multiTF_summary['k562'].keys()

Index(['enhancer_id', 'multiTF_clusters', 'n_multiTF_clusters', 'n_seqlets',
       'n_emvars', 'chrom', 'start', 'end'],
      dtype='object')

In [8]:
# Access a specific enhancer from the summary
# Use the get_enhancer_data() helper to get detailed data on-demand
example_id = multiTF_summary['k562'].iloc[0]['enhancer_id']
print(f"Detailed view of enhancer: {example_id}")
print(f"Multi-TF clusters: {multiTF_summary['k562'].iloc[0]['multiTF_clusters']}")

# Get detailed data using helper function
enhancer_data = get_enhancer_data(raw_data, 'k562', example_id)

print(f"\nCollapsed seqlets ({len(enhancer_data['collapsed_bed'])} intervals):")
display(enhancer_data['collapsed_bed'][['chrom', 'start', 'end', 'representative_tf', 'vierstra_cluster', 'rep_tf_contrib', 'activity_class']])

print(f"\nEmVars ({len(enhancer_data['enhancer_emvars'])} rows) - now with variant_id and skew_pred:")
# Show key columns including the new variant_id and skew_pred
emvar_cols = ['variant_id', 'skew_pred', 'chrom', 'start', 'end', 'representative_tf', 'vierstra_cluster', 'activity_class']
available_cols = [c for c in emvar_cols if c in enhancer_data['enhancer_emvars'].columns]
display(enhancer_data['enhancer_emvars'][available_cols].head(15))

Detailed view of enhancer: EH38E2875836
Multi-TF clusters: AP1/1,Ebox/CACCTG

Collapsed seqlets (13 intervals):


,chrom,start,end,representative_tf,vierstra_cluster,rep_tf_contrib,activity_class
118868,chr1,236097147,236097155,FOSL1,AP1/1,4.934658,Activator
118869,chr1,236097175,236097185,FOSL1,AP1/1,8.690938,Activator
118870,chr1,236097204,236097215,FOSL1,AP1/1,9.492924,Activator
118871,chr1,236097233,236097243,FOSL1,AP1/1,8.404058,Activator
118872,chr1,236097263,236097271,FOSL1,AP1/1,5.491059,Activator
118873,chr1,236097277,236097287,SNAI1,Ebox/CACCTG,-3.956676,Repressor
118874,chr1,236097292,236097300,FOSL1,AP1/1,4.816866,Activator
118875,chr1,236097306,236097316,SNAI1,Ebox/CACCTG,-3.979332,Repressor
118876,chr1,236097321,236097329,FOSL1,AP1/1,5.420013,Activator
118877,chr1,236097349,236097359,FOSL1,AP1/1,8.024785,Activator



EmVars (248 rows) - now with variant_id and skew_pred:


,variant_id,skew_pred,chrom,start,end
11711,chr1:236097148:G:T,-0.722648,chr1,236097147,236097155
11712,chr1:236097149:T:C,-0.883805,chr1,236097147,236097155
11713,chr1:236097149:T:G,-1.156604,chr1,236097147,236097155
11714,chr1:236097149:T:A,-0.990378,chr1,236097147,236097155
11715,chr1:236097150:G:C,-1.097809,chr1,236097147,236097155
11716,chr1:236097150:G:T,-0.589404,chr1,236097147,236097155
11717,chr1:236097150:G:A,-1.101739,chr1,236097147,236097155
11718,chr1:236097151:A:G,-0.964167,chr1,236097147,236097155
11719,chr1:236097151:A:C,-1.178495,chr1,236097147,236097155
11720,chr1:236097151:A:T,-0.971745,chr1,236097147,236097155


In [22]:
get_enhancer_data(raw_data, 'k562', 'EH38E2810579')['collapsed_bed']

,chrom,start,end,name,score,strand,thickStart,thickEnd,full_hit_string,hocomoco_tf,representative_tf,enhancer_id,rep_tf_contrib,vierstra_cluster,activity_class
46806,chr1,50139492,50139497,ERR2_HUMAN.H11MO.0.A_EH38E2810579_K562_-1.0338...,1,ERR2,Repressor,EH38E2810579,ERR2_HUMAN.H11MO.0.A_EH38E2810579_K562_-1.0338...,ERR2_HUMAN.H11MO.0.A,ERR2,EH38E2810579,-1.033847,NR/7,Repressor
46807,chr1,50139512,50139517,TAL1_HUMAN.H11MO.0.A_EH38E2810579_K562_2.87841...,1,TAL1,Activator,EH38E2810579,TAL1_HUMAN.H11MO.0.A_EH38E2810579_K562_2.87841...,TAL1_HUMAN.H11MO.0.A,TAL1,EH38E2810579,2.878414,GATA,Activator
46808,chr1,50139536,50139541,ZNF18_HUMAN.H11MO.0.C_EH38E2810579_K562_-0.995...,1,ZNF18,Repressor,EH38E2810579,ZNF18_HUMAN.H11MO.0.C_EH38E2810579_K562_-0.995...,ZNF18_HUMAN.H11MO.0.C,ZNF18,EH38E2810579,-0.995897,TBX/4,Repressor
46809,chr1,50139541,50139546,NDF1_HUMAN.H11MO.0.A_EH38E2810579_K562_-1.1114...,1,NDF1,Repressor,EH38E2810579,NDF1_HUMAN.H11MO.0.A_EH38E2810579_K562_-1.1114...,NDF1_HUMAN.H11MO.0.A,NDF1,EH38E2810579,-1.111406,Ebox/CAGATGG,Repressor
46810,chr1,50139550,50139558,TYY1_HUMAN.H11MO.0.A_EH38E2810579_K562_-1.9732...,1,TYY1,Repressor,EH38E2810579,TYY1_HUMAN.H11MO.0.A_EH38E2810579_K562_-1.9732...,TYY1_HUMAN.H11MO.0.A,TYY1,EH38E2810579,-1.973218,YY1,Repressor
46811,chr1,50139559,50139567,FOXQ1_HUMAN.H11MO.0.C_EH38E2810579_K562_3.0919...,2,GATA6,Activator,EH38E2810579,GATA6_HUMAN.H11MO.0.A_EH38E2810579_K562_6.0960...,GATA6_HUMAN.H11MO.0.A,GATA6,EH38E2810579,6.096037,GATA,Activator
46812,chr1,50139646,50139655,ZEB1_HUMAN.H11MO.0.A_EH38E2810579_K562_-2.7660...,1,ZEB1,Repressor,EH38E2810579,ZEB1_HUMAN.H11MO.0.A_EH38E2810579_K562_-2.7660...,ZEB1_HUMAN.H11MO.0.A,ZEB1,EH38E2810579,-2.766059,SNAI2,Repressor


In [24]:
all_variants[all_variants['enhancer_ids'] == 'EH38E2810579']

,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF
17342,chr1:50139496:C:G,chr1,50139496,C,G,k562,0.749094,EH38E2810579,True
17343,chr1:50139513:T:A,chr1,50139513,T,A,k562,-0.737187,EH38E2810579,True
17344,chr1:50139513:T:C,chr1,50139513,T,C,k562,-0.687216,EH38E2810579,True
17345,chr1:50139513:T:G,chr1,50139513,T,G,k562,-0.730143,EH38E2810579,True
17346,chr1:50139514:A:C,chr1,50139514,A,C,k562,-0.713422,EH38E2810579,True
17347,chr1:50139514:A:G,chr1,50139514,A,G,k562,-0.699427,EH38E2810579,True
17348,chr1:50139514:A:T,chr1,50139514,A,T,k562,-0.656112,EH38E2810579,True
17349,chr1:50139515:T:A,chr1,50139515,T,A,k562,-0.755638,EH38E2810579,True
17350,chr1:50139515:T:C,chr1,50139515,T,C,k562,-0.818893,EH38E2810579,True
17351,chr1:50139515:T:G,chr1,50139515,T,G,k562,-0.779546,EH38E2810579,True


In [25]:
get_enhancer_data(raw_data, 'k562', 'EH38E2875903')['collapsed_bed']

,chrom,start,end,name,score,strand,thickStart,thickEnd,full_hit_string,hocomoco_tf,representative_tf,enhancer_id,rep_tf_contrib,vierstra_cluster,activity_class
118949,chr1,236163498,236163509,GFI1_HUMAN.H11MO.0.C_EH38E2875903_K562_-7.4676...,1,GFI1,Repressor,EH38E2875903,GFI1_HUMAN.H11MO.0.C_EH38E2875903_K562_-7.4676...,GFI1_HUMAN.H11MO.0.C,GFI1,EH38E2875903,-7.467690,GFI,Repressor
118950,chr1,236163522,236163527,ZN816_HUMAN.H11MO.0.C_EH38E2875903_K562_2.8079...,1,ZN816,Activator,EH38E2875903,ZN816_HUMAN.H11MO.0.C_EH38E2875903_K562_2.8079...,ZN816_HUMAN.H11MO.0.C,ZN816,EH38E2875903,2.807947,MZF1,Activator
118951,chr1,236163541,236163552,ZN281_HUMAN.H11MO.0.A_EH38E2875903_K562_10.378...,1,ZN281,Activator,EH38E2875903,ZN281_HUMAN.H11MO.0.A_EH38E2875903_K562_10.378...,ZN281_HUMAN.H11MO.0.A,ZN281,EH38E2875903,10.378629,KLF/SP/2,Activator
118952,chr1,236163559,236163564,EHF_HUMAN.H11MO.0.B_EH38E2875903_K562_2.929543...,1,EHF,Activator,EH38E2875903,EHF_HUMAN.H11MO.0.B_EH38E2875903_K562_2.929543...,EHF_HUMAN.H11MO.0.B,EHF,EH38E2875903,2.929544,ETS/2,Activator
118953,chr1,236163570,236163575,GFI1_HUMAN.H11MO.0.C_EH38E2875903_K562_-1.0688...,1,GFI1,Repressor,EH38E2875903,GFI1_HUMAN.H11MO.0.C_EH38E2875903_K562_-1.0688...,GFI1_HUMAN.H11MO.0.C,GFI1,EH38E2875903,-1.068862,GFI,Repressor
118954,chr1,236163621,236163626,MAFG_HUMAN.H11MO.0.A_EH38E2875903_K562_-1.0087...,1,MAFG,Repressor,EH38E2875903,MAFG_HUMAN.H11MO.0.A_EH38E2875903_K562_-1.0087...,MAFG_HUMAN.H11MO.0.A,MAFG,EH38E2875903,-1.008767,MAF,Repressor


In [28]:
all_variants[(all_variants['enhancer_ids'] == 'EH38E2875903') & (all_variants['cell_type'] == 'k562')]

,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF
2638,chr1:236163501:A:C,chr1,236163501,A,C,k562,0.689453,EH38E2875903,True
2639,chr1:236163501:A:G,chr1,236163501,A,G,k562,0.746263,EH38E2875903,True
2640,chr1:236163501:A:T,chr1,236163501,A,T,k562,1.114761,EH38E2875903,True
2641,chr1:236163502:A:C,chr1,236163502,A,C,k562,1.272333,EH38E2875903,True
2642,chr1:236163502:A:G,chr1,236163502,A,G,k562,1.466583,EH38E2875903,True
...,...,...,...,...,...,...,...,...,...
2707,chr1:236163571:A:C,chr1,236163571,A,C,k562,0.675378,EH38E2875903,True
2708,chr1:236163572:A:C,chr1,236163572,A,C,k562,0.551287,EH38E2875903,True
2709,chr1:236163572:A:G,chr1,236163572,A,G,k562,0.637488,EH38E2875903,True
2710,chr1:236163624:A:G,chr1,236163624,A,G,k562,0.883473,EH38E2875903,True


In [9]:
# Combine all cell types into one dataframe for downstream annotation
all_variants = pd.concat([
    raw_data['k562']['variants'],
    raw_data['hepg2']['variants'],
    raw_data['sknsh']['variants']
], ignore_index=True)

print(f"Total variants across all cell types: {len(all_variants)}")
print(f"Unique variant IDs: {all_variants['variant_id'].nunique()}")
print(f"\nBreakdown by cell type:")
print(all_variants.groupby('cell_type').size())
print(f"\nVariants in multi-TF enhancers by cell type:")
print(all_variants[all_variants['is_multiTF']].groupby('cell_type').size())

Total variants across all cell types: 49489
Unique variant IDs: 31832

Breakdown by cell type:
cell_type
hepg2    14464
k562     17850
sknsh    17175
dtype: int64

Variants in multi-TF enhancers by cell type:
cell_type
hepg2    2861
k562     3589
sknsh    3953
dtype: int64


In [10]:
# Column structure for downstream annotation
print("Columns available for downstream annotation:")
print(all_variants.columns.tolist())
print("\nSample of variant_id format:")
print(all_variants['variant_id'].head())

# Optional: Export to TSV for downstream annotation
# all_variants.to_csv('multiTF_emvars_for_annotation.tsv', sep='\t', index=False)

Columns available for downstream annotation:
['variant_id', 'chrom', 'pos', 'ref', 'alt', 'cell_type', 'skew_pred', 'enhancer_ids', 'is_multiTF']

Sample of variant_id format:
0    chr1:235709267:G:C
1    chr1:235715990:T:G
2    chr1:235715991:G:A
3    chr1:235715991:G:C
4    chr1:235715991:G:T
Name: variant_id, dtype: object
